In [4]:
import pandas as pd

In [5]:
df =pd.read_csv('Sample - Superstore.csv',
  encoding='latin-1');
print('Regions:', df['Region'].unique()); print('Region counts:\n',
df['Region'].value_counts()); 
print('Total rows:', len(df))

Regions: <StringArray>
['South', 'West', 'Central', 'East']
Length: 4, dtype: str
Region counts:
 Region
West       3203
East       2848
Central    2323
South      1620
Name: count, dtype: int64
Total rows: 9994


"""
Data Preparation
Splits Superstore dataset into 4 regional CSVs with:
  - Different column schemas per region
  - Cross-file duplicate Row IDs (West rows injected into East)
  - Missing sales values (injected into Central and South)
Outputs files to Requirements/environment/input_artifacts/
"""

In [6]:
import pandas as pd
import json
import os

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_FILE  = 'Sample - Superstore.csv'
OUTPUT_DIR  = '../Requirements/environment/input_artifacts'
ORACLE_OUT  = '../Requirements/environment/oracle_expected.json'

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [7]:
# ── Load & keep only needed columns ───────────────────────────────────────
df = pd.read_csv(INPUT_FILE, encoding='latin-1')
df = df[['Row ID', 'Sub-Category', 'Sales', 'Quantity', 'Order Date', 'Region']].copy()

# ── Split by region ────────────────────────────────────────────────────────
west    = df[df['Region'] == 'West'].drop(columns='Region').reset_index(drop=True)
east    = df[df['Region'] == 'East'].drop(columns='Region').reset_index(drop=True)
central = df[df['Region'] == 'Central'].drop(columns='Region').reset_index(drop=True)
south   = df[df['Region'] == 'South'].drop(columns='Region').reset_index(drop=True)

In [8]:
# ── Inject cross-file duplicates ───────────────────────────────────────────

# Inject 8 West rows into East (duplicate Row IDs across files)
cross_dups = west.head(8).copy()
east = pd.concat([east, cross_dups], ignore_index=True)

# Duplicate 4 rows inside Central itself
intra_dups = central.iloc[[2, 7, 14, 20]].copy()
central = pd.concat([central, intra_dups], ignore_index=True)

N_DUPLICATES = 12


In [9]:
# ── Inject missing Sales values ────────────────────────────────────────────
central_missing_idx = [10, 40, 150, 640, 1233, 1456, 1873, 1919, 2187, 2255]            # 10 rows
south_missing_idx   = [5, 230, 596, 854, 1476]                                          # 5 rows
west_missing_idx    = [13, 87, 123, 476, 798, 1024, 1356, 1689, 2012, 2682, 3121]       # 11 rows
east_missing_idx    = [39, 156, 450, 789, 1123, 1456, 1789, 2123, 2456, 2789]           # 10 rows

west.loc[west_missing_idx, 'Sales']    = None
east.loc[east_missing_idx, 'Sales'] = None 
central.loc[central_missing_idx, 'Sales'] = None
south.loc[south_missing_idx, 'Sales']        = None
N_MISSING = len(central_missing_idx) + len(south_missing_idx) + len(west_missing_idx) + len(east_missing_idx)   # 36 rows


In [10]:
# ── Inject Negative Sales values ───────────
west.loc[[100, 230, 743, 1150, 1456, 2343, 2890], 'Sales']    = -44.67             # 7 negative
east.loc[[80, 150], 'Sales'] = -9.99                                               # 2 negative
central.loc[[30, 800, 1200, 1600, 2090], 'Sales']  = -15.50                        # 5 negative
south.loc[[8, 489, 1344], 'Sales']     = -20.00                                    # 3 negative

In [11]:
# ── Inject White Space in Sub-Category ───────────────────────────────────────────────
west.loc[[0, 5, 12], 'Sub-Category']  = west.loc[[0, 5, 12], 'Sub-Category'].str.strip().apply(lambda x:
'  ' + x)
south.loc[[1, 9], 'Sub-Category']     = south.loc[[1, 9], 'Sub-Category'].str.strip().apply(lambda x: x
+ ' ')

In [12]:
# ── Rename columns differently per region (inconsistent schemas) ───────────
west = west.rename(columns={
    'Row ID':       'row_id',
    'Sub-Category': 'sub_category',
    'Sales':        'sales',
    'Quantity':     'quantity',
    'Order Date':   'order_date',
})

east = east.rename(columns={
    'Row ID':       'transaction_id',
    'Sub-Category': 'product_category',
    'Sales':        'revenue',
    'Quantity':     'units',
    'Order Date':   'date',
})

central = central.rename(columns={
    'Row ID':       'txn_id',
    'Sub-Category': 'category',
    'Sales':        'sales_amount',
    'Quantity':     'qty',
    'Order Date':   'sale_date',
})

south = south.rename(columns={
    'Row ID':       'id',
    'Sub-Category': 'item_category',
    'Sales':        'amount',
    'Quantity':     'num_units',
    'Order Date':   'transaction_date',
})


In [13]:
# ── Save CSV files ─────────────────────────────────────────────────────────
west.to_csv(   os.path.join(OUTPUT_DIR, 'sales_west.csv'),    index=False)
east.to_csv(   os.path.join(OUTPUT_DIR, 'sales_east.csv'),    index=False)
central.to_csv(os.path.join(OUTPUT_DIR, 'sales_central.csv'), index=False)
south.to_csv(  os.path.join(OUTPUT_DIR, 'sales_south.csv'),   index=False)

print("=== Files Created ===")
print(f"  sales_west.csv    : {len(west)} rows")
print(f"  sales_east.csv    : {len(east)} rows  (+{N_DUPLICATES} duplicate rows from West injected)")
print(f"  sales_central.csv : {len(central)} rows  ({len(central_missing_idx)} missing sales values)")
print(f"  sales_south.csv   : {len(south)} rows  ({len(south_missing_idx)} missing sales values)")


=== Files Created ===
  sales_west.csv    : 3203 rows
  sales_east.csv    : 2856 rows  (+12 duplicate rows from West injected)
  sales_central.csv : 2327 rows  (10 missing sales values)
  sales_south.csv   : 1620 rows  (5 missing sales values)


In [14]:
SCHEMA_MAP = [
    ('sales_west.csv',    'row_id',         'sub_category',    'sales',        'west'),
    ('sales_east.csv',    'transaction_id', 'product_category','revenue',      'east'),
    ('sales_central.csv', 'txn_id',         'category',        'sales_amount', 'central'),
    ('sales_south.csv',   'id',             'item_category',   'amount',       'south'),
]

frames = []
total_missing = 0

In [15]:
for fname, id_col, cat_col, sales_col, region in SCHEMA_MAP:
      d = pd.read_csv(os.path.join(OUTPUT_DIR, fname))
      d = d.rename(columns={id_col: 'row_id', cat_col: 'sub_category', sales_col: 'sales'})
      d = d[['row_id', 'sub_category', 'sales']].copy()

      # Count both null AND negative as invalid
      invalid = int(d['sales'].isna().sum()) + int((d['sales'] < 0).sum())
      total_missing += invalid

      # Drop null and negative rows
      d = d.dropna(subset=['sales'])
      d = d[d['sales'] > 0]

      d['region'] = region
      frames.append(d)

combined  = pd.concat(frames, ignore_index=True)
before    = len(combined)
combined  = combined.drop_duplicates(subset=['row_id'], keep='first')
after     = len(combined)
dups_removed = before - after

print("\n=== Combined DataFrame ===")
print(f"Total rows (with duplicates): {before}")
print(f"Total rows (after removing duplicates): {after}")
print(f"Total duplicate rows removed: {dups_removed}")


=== Combined DataFrame ===
Total rows (with duplicates): 9953
Total rows (after removing duplicates): 9941
Total duplicate rows removed: 12


In [17]:
region_totals = {
    r: round(float(combined[combined['region'] == r]['sales'].sum()), 2)
    for r in ['west', 'east', 'central', 'south']
}
total_sales = round(float(combined['sales'].sum()), 2)
top3 = list(
    combined.groupby('sub_category')['sales']
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index
)

print("\n=== Total Sales per Region ===")
for region, total in region_totals.items():
    print(f"{region.capitalize()}: ${total:,}")

print(f"\n=== Overall Total Sales ===")
print(f"Total Sales: ${total_sales:,}")

print(f"\n=== Top 3 Sub-Categories by Sales ===")
for i, sub_category in enumerate(top3, start=1):
    print(f"{i}. {sub_category}")


=== Total Sales per Region ===
West: $722,066.44
East: $674,463.32
Central: $498,656.98
South: $391,403.5

=== Overall Total Sales ===
Total Sales: $2,286,590.24

=== Top 3 Sub-Categories by Sales ===
1. Phones
2. Chairs
3. Storage


In [18]:
oracle = {
    "total_sales": total_sales,
    "region_totals": region_totals,
    "top_3_sub_categories": top3,
    "duplicate_rows_removed": int(dups_removed),
    "records_skipped_missing_sales": int(total_missing),
}

print("\n=== Expected Oracle Output ===")
print(json.dumps(oracle, indent=2))

with open(ORACLE_OUT, 'w') as f:
    json.dump(oracle, f, indent=2)
print(f"\nOracle saved to: {ORACLE_OUT}")


=== Expected Oracle Output ===
{
  "total_sales": 2286590.24,
  "region_totals": {
    "west": 722066.44,
    "east": 674463.32,
    "central": 498656.98,
    "south": 391403.5
  },
  "top_3_sub_categories": [
    "Phones",
    "Chairs",
    "Storage"
  ],
  "duplicate_rows_removed": 12,
  "records_skipped_missing_sales": 53
}

Oracle saved to: D:/Work/SwarmBench/SwarmBench_DataAnalysis/initial_data/oracle_expected.json


In [ ]:
!pip install faker

# Advanced SwarmBench Data Preparation Script

import pandas as pd
import numpy as np
import random
import os
import json
from faker import Faker

fake = Faker()
random.seed(42)
np.random.seed(42)

# ============================================================
# CONFIG
# ============================================================



REGIONS = ["west", "east", "central", "south"]
QUARTERS = ["q1", "q2", "q3", "q4"]

ROWS_PER_FILE = 1200
DUPLICATE_RATE = 0.08
INVALID_RATE = 0.06

# ============================================================
# COLUMN NAME VARIANTS
# ============================================================

ID_VARIANTS = [
    "row_id",
    "transaction_id",
    "txn_id",
    "sale_id",
    "record_id",
    "id"
]

CATEGORY_VARIANTS = [
    "sub_category",
    "subcategory",
    "sub_cat",
    "product_category",
    "item_type",
    "category"
]

SALES_VARIANTS = [
    "sales",
    "sales_amount",
    "revenue",
    "gross_sales",
    "usd_amount",
    "amount"
]

# ============================================================
# NOISE HELPERS
# ============================================================


def random_sales_format(value):
    """Inject dirty formatting into numeric sales values."""

    formats = [
        lambda x: f"{x:.2f}",
        lambda x: f" ${x:,.2f} ",
        lambda x: f"{x:,.2f}",
        lambda x: f" {x} ",
        lambda x: f"USD {x:.2f}",
    ]

    return random.choice(formats)(value)



INVALID_VALUES = [
    "",
    None,
    "N/A",
    "NULL",
    "-",
    "missing",
    "error",
    "INVALID",
    "-10",
    "-50",
    "-999",
    "USD -45",
    "unknown"
]



def dirty_subcategory(value):
    """Inject casing and whitespace inconsistencies."""

    variants = [
        value,
        value.lower(),
        value.upper(),
        f" {value}",
        f"{value} ",
        f" {value} ",
    ]

    return random.choice(variants)


# ============================================================
# LOAD SOURCE DATA
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading Superstore dataset...")

df = pd.read_csv(INPUT_FILE, encoding="latin1")

# Normalize source columns
source = pd.DataFrame({
    "row_id": df["Row ID"],
    "sub_category": df["Sub-Category"],
    "sales": df["Sales"]
})

source = source.dropna(subset=["row_id", "sub_category", "sales"])

print(f"Loaded {len(source)} source rows")

# ============================================================
# GENERATE SHARDS
# ============================================================

all_duplicate_ids = []
all_invalid_count = 0
all_duplicate_count = 0

file_metadata = []

for region in REGIONS:
    for quarter in QUARTERS:

        shard_name = f"{region}_{quarter}"
        print(f"Generating {shard_name}...")

        shard = source.sample(ROWS_PER_FILE, replace=True).copy()

        # --------------------------------------------------------
        # Introduce duplicates across files
        # --------------------------------------------------------

        duplicate_count = int(ROWS_PER_FILE * DUPLICATE_RATE)

        if all_duplicate_ids:
            duplicate_ids = random.sample(
                all_duplicate_ids,
                min(len(all_duplicate_ids), duplicate_count)
            )

            for i in range(len(duplicate_ids)):
                shard.iloc[i, shard.columns.get_loc("row_id")] = duplicate_ids[i]

            all_duplicate_count += len(duplicate_ids)

        new_ids = list(shard["row_id"].sample(duplicate_count))
        all_duplicate_ids.extend(new_ids)

        # --------------------------------------------------------
        # Dirty subcategory formatting
        # --------------------------------------------------------

        shard["sub_category"] = shard["sub_category"].apply(dirty_subcategory)

        # --------------------------------------------------------
        # Dirty sales formatting
        # --------------------------------------------------------

        shard["sales"] = shard["sales"].apply(random_sales_format)

        # --------------------------------------------------------
        # Inject invalid rows
        # --------------------------------------------------------

        invalid_count = int(ROWS_PER_FILE * INVALID_RATE)

        invalid_indices = random.sample(range(len(shard)), invalid_count)

        for idx in invalid_indices:
            shard.iloc[idx, shard.columns.get_loc("sales")] = random.choice(INVALID_VALUES)

        all_invalid_count += invalid_count

        # --------------------------------------------------------
        # Randomize schema names
        # --------------------------------------------------------

        id_col = random.choice(ID_VARIANTS)
        category_col = random.choice(CATEGORY_VARIANTS)
        sales_col = random.choice(SALES_VARIANTS)

        renamed = shard.rename(columns={
            "row_id": id_col,
            "sub_category": category_col,
            "sales": sales_col
        })

        # --------------------------------------------------------
        # Add noisy irrelevant columns
        # --------------------------------------------------------

        renamed["customer_notes"] = [fake.sentence() for _ in range(len(renamed))]
        renamed["shipping_comment"] = [fake.text(max_nb_chars=40) for _ in range(len(renamed))]
        renamed["internal_code"] = np.random.randint(1000, 9999, size=len(renamed))
        renamed["processed_by"] = [fake.first_name() for _ in range(len(renamed))]

        # --------------------------------------------------------
        # Shuffle columns randomly
        # --------------------------------------------------------

        cols = list(renamed.columns)
        random.shuffle(cols)
        renamed = renamed[cols]

        # --------------------------------------------------------
        # Shuffle row order
        # --------------------------------------------------------

        renamed = renamed.sample(frac=1).reset_index(drop=True)

        # --------------------------------------------------------
        # Save CSV
        # --------------------------------------------------------

        output_path = os.path.join(
            OUTPUT_DIR,
            f"sales_{region}_{quarter}.csv"
        )

        renamed.to_csv(output_path, index=False)

        file_metadata.append({
            "file": output_path,
            "id_column": id_col,
            "category_column": category_col,
            "sales_column": sales_col,
            "rows": len(renamed)
        })

# ============================================================
# SAVE METADATA
# ============================================================

metadata_path = os.path.join(OUTPUT_DIR, "schema_reference.json")

with open(metadata_path, "w") as f:
    json.dump(file_metadata, f, indent=2)

# ============================================================
# SUMMARY
# ============================================================

print("\n================================================")
print("DATA GENERATION COMPLETE")
print("================================================")
print(f"Generated files: {len(file_metadata)}")
print(f"Approx duplicate rows injected: {all_duplicate_count}")
print(f"Approx invalid rows injected: {all_invalid_count}")
print(f"Output directory: {OUTPUT_DIR}")
print("================================================")


Loading Superstore dataset...
Loaded 9994 source rows
Generating west_q1...
Generating west_q2...
Generating west_q3...
Generating west_q4...
Generating east_q1...
Generating east_q2...
Generating east_q3...
Generating east_q4...
Generating central_q1...
Generating central_q2...
Generating central_q3...
Generating central_q4...
Generating south_q1...
Generating south_q2...
Generating south_q3...
Generating south_q4...

DATA GENERATION COMPLETE
Generated files: 16
Approx duplicate rows injected: 1440
Approx invalid rows injected: 1152
Output directory: ../Requirements/environment/input_artifacts


   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   -------------------- ------------------- 1.0/2.0 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 3.2 MB/s  0:00:01


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
